# Exploratory Data Analysis (EDA)

In this phase, we analyze the statistical properties of the bestiary dataset to uncover patterns, identify relationships, and validate our hypotheses about monster power levels. The goal is to move from raw data to actionable insights by examining how individual features—such as Hit Points, Armor Class, and Ability Scores—interact with the target variable, **Challenge Rating (CR)**. This process ensures that the features we eventually feed into our machine learning model provide a clear and consistent signal for predicting combat efficiency.

**Sources Used:**
1. [Steps for Mastering Exploratory Data Analysis](https://www.geeksforgeeks.org/data-analysis/steps-for-mastering-exploratory-data-analysis-eda-steps/) (GeeksforGeeks)
2. [A Data Scientist's Essential Guide to EDA](https://medium.com/data-science/a-data-scientists-essential-guide-to-exploratory-data-analysis-25637eee0cf6) (Medium)
3. [What is Exploratory Data Analysis?](https://www.ibm.com/think/topics/exploratory-data-analysis) (IBM)

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('fivethirtyeight')
sns.set_theme(style="whitegrid")

# Load prepared dataset
prepared_df = pd.read_csv('../../data/proposed/prepared_bestiary.csv')

# Print dataset info
print("--- Dataset Information ---")
prepared_df.info()

# Display summary statistics
print("\n--- Statistical Summary ---")
display(prepared_df.describe())

# Display first 5 records
print("\n--- First 5 Records ---")
display(prepared_df.head())

## Phase 1: Univariate Analysis (Individual Distributions)
**Goal:** Understand the "shape" of the data and identify outliers or variance issues before modeling.

### 1.1 Numerical Histograms
We plot histograms to determine if numeric features are normally distributed or skewed.

* **Target Variable:** `CR` (Challenge Rating) — Expected to have a heavy skew toward lower values.
* **Defensive Stats:** `AC` and `HP`.
* **Attributes:** The six Core Ability Scores (`STR`, `DEX`, `CON`, `INT`, `WIS`, `CHA`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visual style
plt.style.use('fivethirtyeight')

# Plotting the distribution of CR
plt.figure(figsize=(10, 6))
sns.histplot(prepared_df['CR'], bins=30, kde=True, color='teal')

# Customizing the labels
plt.title('Distribution of Challenge Rating (CR)')
plt.xlabel('Challenge Rating')
plt.ylabel('Frequency of Monsters')

# Save the visualization for the poster
plt.savefig('cr_distribution.png')
plt.show()

# Display summary statistics
print(prepared_df['CR'].describe())

This distribution shows that your dataset is heavily "bottom-heavy," meaning most monsters are weak while very few are legendary. The average Challenge Rating is around 6, but the median is actually 4, which confirms that a large cluster of monsters sits between CR 0 and CR 5. Your standard deviation of 6.18 is quite high relative to the mean, indicating a wide spread in power levels across the rest of the list.

The sharp peak at the low end of the chart suggests that the game design focuses on common threats that players face often, while the long tail stretching toward CR 30 represents rare boss-level creatures. Because the data is so skewed, your model will have plenty of examples of weak monsters to learn from, but it might struggle to predict high CR values accurately unless you specifically account for those sparse data points at the top end.

In [ ]:
# Set visual style
plt.style.use('fivethirtyeight')

# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot AC distribution
ax1.hist(prepared_df['AC'], bins=15, color='#30a2da', edgecolor='black', alpha=0.8)
ax1.set_title('Distribution of Armor Class (AC)', fontsize=16)
ax1.set_xlabel('AC Value')
ax1.set_ylabel('Number of Monsters')

# Plot HP distribution
ax2.hist(prepared_df['HP'], bins=30, color='#fc4f30', edgecolor='black', alpha=0.8)
ax2.set_title('Distribution of Hit Points (HP)', fontsize=16)
ax2.set_xlabel('HP Value')
ax2.set_ylabel('Number of Monsters')

# Save plot
plt.tight_layout()
plt.savefig('defensive_stats_distribution.png')

# Print summary statistics
print(prepared_df[['AC', 'HP']].describe())

Armor Class is relatively balanced and predictable, while Hit Points are extreme and varied.

Most monsters have an AC between 12 and 17, creating a bell-shaped curve that centers around 15. Since the maximum AC is only 25, the gap between the weakest and strongest monster is small. This suggests that in your model, a single point of AC will carry significant weight because the total range is so narrow.

Hit Points tell a completely different story. The data is extremely stretched out, with a massive cluster of low-health monsters and a few "sponges" reaching over 700 HP. The standard deviation is nearly 100, which is larger than the average itself. This means "average" isn't a very helpful description for HP; instead, you have a huge variety of health pools that likely scale up very quickly as Challenge Rating increases.

Because HP has such a massive range (1 to 725) compared to AC (5 to 25), your model might naturally pay more attention to HP.

In [ ]:
import matplotlib.pyplot as plt

# Define the attributes to plot
attributes = ['Strength', 'Dexterity', 'Constitution', 'Intelligence', 'Wisdom', 'Charisma']
colors = ['#e5ae38', '#6d904f', '#8b8b8b', '#810f7c', '#008fd5', '#e5ae37']

# Set the visual style
plt.style.use('fivethirtyeight')

# Create a 2x3 grid of subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(attributes):
    axes[i].hist(prepared_df[col], bins=15, color=colors[i], edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}', fontsize=14)
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')
    # Draw a vertical line at 10 (Human Average)
    axes[i].axvline(10, color='red', linestyle='--', linewidth=1, label='Human Avg (10)')

plt.tight_layout()

# Save the plot for the poster
plt.savefig('attributes_distribution.png')
plt.show()

# Display summary statistics for the attributes
print(prepared_df[attributes].describe())

Monster stats follow clear "biological" patterns where physical traits are consistently higher than mental traits.

Most monsters are physically stronger and heartier than the average human, as seen by the peaks for Strength and Constitution sitting well to the right of your red "Human Average" line. Strength, in particular, has a very broad spread, meaning it varies wildly from tiny weaklings to massive giants. Intelligence and Charisma, however, center much closer to the human baseline of 10, with a large number of monsters actually falling below it.

From a modeling perspective, Strength and Constitution will likely be your strongest predictors for Challenge Rating. Because they have the highest means and the most significant variance, they represent the "power scaling" of a creature better than Dexterity or Wisdom, which stay relatively cramped in a narrow range for most of the bestiary.

### 1.2 Categorical Counts
We use bar charts to visualize the frequency of different monster groups:
* **Size:** Distribution from Tiny to Gargantuan.
* **Type:** Distribution across categories like Undead, Dragon, Fiend, etc.

In [ ]:
import matplotlib.pyplot as plt

# Set visual style
plt.style.use('fivethirtyeight')

# Define size order
size_order_lower = ['tiny', 'small', 'medium', 'large', 'huge', 'gargantuan']

# Plot distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=prepared_df, x='Size', order=size_order_lower, palette='viridis', hue='Size', legend=False)

# Set labels
plt.title('Distribution of Monster Sizes', fontsize=16)
plt.xlabel('Size Category')
plt.ylabel('Number of Monsters')
plt.show()

# Print counts
print(prepared_df['Size'].value_counts().reindex(size_order_lower))

Medium is the dominant standard for monster design, accounting for more than half of the entire bestiary.

This distribution shows that the "Medium" category is the baseline for D&D encounters, likely because most player characters are also medium-sized. As monsters get larger or smaller than this midpoint, their frequency drops off significantly. Interestingly, there are more "Gargantuan" monsters than "Tiny" ones, suggesting the game provides more high-scale bosses than it does microscopic threats.

From a modeling perspective, the "Medium" category is your most reliable data point, but the "Large" and "Huge" categories still have enough samples to be statistically significant. However, the extremes—Tiny and Gargantuan—are relatively rare. This means your model might become very good at predicting the CR of human-sized enemies but will have much less "experience" with the outliers at either end of the size spectrum.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('fivethirtyeight')

# Calculate type frequency
type_counts = prepared_df['Type'].value_counts()

# Plot distribution
plt.figure(figsize=(12, 8))
sns.barplot(x=type_counts.values, y=type_counts.index, hue=type_counts.index, palette='magma', legend=False)

# Set labels
plt.title('Distribution of Monster Types', fontsize=16)
plt.xlabel('Number of Monsters')
plt.ylabel('Creature Type')

# Save plot
plt.tight_layout()
plt.savefig('type_distribution.png')
plt.show()

# Print counts
print(type_counts)

Monster types are extremely lopsided, with humanoids acting as the primary population of the game world while many other categories remain niche.

The "Humanoid" type is a massive outlier, containing over three times as many entries as the next closest category. This makes sense for a game centered on social interaction and urban combat, but for your model, it means the features that define a humanoid will be learned much more thoroughly than those of an "Ooze" or a "Plant." The "Big Three" of monster design—Humanoids, Monstrosities, and Undead—make up the bulk of the dataset, providing the most reliable patterns for your analysis.

You also have a "long tail" of swarms and hybrid types (like celestial or fiend) that appear only a few times. These rare types are essentially "noise" for a machine learning model because there aren't enough examples for it to find a consistent trend. If you decide to group all "Swarm of..." entries into a single "Swarm" category during your feature engineering, you might help the model make better sense of these low-frequency creatures.

### 1.3 Outlier Detection & Variance Check
* **Boxplots:** We generate boxplots for `HP` and `AC` to identify extreme statistical outliers (e.g., the Tarrasque).
* **Variance Check:** We identify features with "near-zero variance." If a stat like `Speed` or `Intelligence` is identical for 95% of the dataset, it may offer little predictive value for the model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('fivethirtyeight')

# Setup subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot AC boxplot
sns.boxplot(x=prepared_df['AC'], ax=ax1, color='#30a2da', flierprops={'markerfacecolor':'red', 'marker':'o'})
ax1.set_title('Outlier Check: Armor Class (AC)', fontsize=16)
ax1.set_xlabel('AC Value')

# Plot HP boxplot
sns.boxplot(x=prepared_df['HP'], ax=ax2, color='#fc4f30', flierprops={'markerfacecolor':'red', 'marker':'o'})
ax2.set_title('Outlier Check: Hit Points (HP)', fontsize=16)
ax2.set_xlabel('HP Value')

plt.tight_layout()
plt.show()

# Print statistical outliers
print("--- Top 5 HP Outliers ---")
display(prepared_df.nlargest(5, 'HP')[['Name', 'CR', 'HP']])

print("\n--- Top 5 AC Outliers ---")
display(prepared_df.nlargest(5, 'AC')[['Name', 'CR', 'AC']])

This data shows how D&D monster stats are designed with specific limits. Armor Class is a bounded stat. Most monsters have an AC between 12 and 18. Even the most powerful legendary creatures rarely go above 25. This means your model only needs to look at a small range of numbers to understand a monster's defense. Because the range is so small, every single point of AC matters a lot for the final Challenge Rating.

Hit Points are unbounded and scale differently. While most monsters stay under 100 HP, the top tier bosses jump to over 600 or 700. This creates a very long tail in your data. The gap between a normal monster and a boss is massive in terms of health but small in terms of AC. This tells the model that HP is the primary way the game scales difficulty at higher levels.

The top 5 lists prove this pattern. All of the strongest monsters hit the same ceiling of 25 for AC, but their HP totals vary by much larger amounts. When you build your model, it will likely rely on HP to distinguish between different high level monsters since their AC values will all look very similar.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('fivethirtyeight')
sns.set_theme(style="whitegrid")

# Identify numeric features
numeric_features = prepared_df.select_dtypes(include=[np.number]).columns.tolist()

# Normalize data (Min-Max Scaling)
normalized_df = (prepared_df[numeric_features] - prepared_df[numeric_features].min()) / \
                (prepared_df[numeric_features].max() - prepared_df[numeric_features].min() + 1e-9)

# Create KDE plot
plt.figure(figsize=(12, 6))
for col in numeric_features:
    sns.kdeplot(data=normalized_df[col], label=col, bw_adjust=0.5)

# Set labels and legend
plt.title('Feature Concentration: Density Comparison', fontsize=16)
plt.xlabel('Normalized Value (0 to 1)')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
plt.tight_layout()
plt.show()

# Print feature list
print("\n--- Numeric Features Identified ---")
print(numeric_features)

This density plot shows which monster stats are predictable and which are varied. Stats with a high, thin peak have low variance. This means most monsters share nearly the same value for that trait. Stats with a flat, wide curve have high variance, meaning the values are spread out across the bestiary.

Dexterity, Wisdom, and Armor Class show the sharpest peaks. This indicates these stats stay very consistent regardless of how powerful a monster is. Most creatures in the game cluster around the same average for these three traits. Because they do not change much, they may have less influence on the final Challenge Rating.

Strength and Hit Points show much flatter curves. These stats are spread across the entire range from zero to one. This variety suggests that as monsters get stronger, these are the primary numbers that designers change. These high-variance features are likely the most important signals for your model to use when predicting a monster's power level.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Setup dominance data
cols = ['AC', 'HP', 'Strength', 'Dexterity', 'Constitution', 'Intelligence', 'Wisdom', 'Charisma', 'CR']
dom = pd.Series({c: prepared_df[c].value_counts(normalize=True).max() * 100 for c in cols}).sort_values(ascending=False)

# Plot feature dominance
plt.figure(figsize=(10, 6))
colors = ['#fc4f30' if x > 80 else '#30a2da' for x in dom]
ax = dom.plot(kind='bar', color=colors, edgecolor='black', width=0.8)

# Add value labels and threshold lines
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2., p.get_height() + 1), ha='center', fontweight='bold')

plt.axhline(95, color='red', ls='--', label='NZV (95%)')
plt.axhline(80, color='orange', ls='--', label='Warning (80%)')
plt.title('Feature Dominance (%)', fontsize=15)
plt.ylim(0, 110)
plt.legend()
plt.show()

# Print raw values
print(dom.round(2))

This chart confirms that your data is incredibly diverse and healthy for machine learning.

Dexterity and Wisdom are your most "repetitive" stats, but even they only have a common value about 19% of the time. This means that for 80% of the monsters in your list, these stats are varied and unique. Since no bar comes close to the 80% or 95% lines, you do not have a problem with near-zero variance. Every stat you have chosen offers unique information that the model can use to distinguish one monster from another.

Hit Points (HP) has the lowest dominance at only 3.5%. This is the best possible result for a predictive model. It means almost every monster has a specific, unique health total. This lack of repetition creates a very smooth gradient of data points, which helps a model find the exact mathematical "tipping point" where a monster moves from one Challenge Rating to the next.

## Phase 2: Bivariate Analysis (Direct Correlations)
**Goal:** Identify which specific numerical attributes move in sync with the Challenge Rating.

### 2.1 Correlation Heatmap
We generate a heatmap to calculate the correlation coefficients between `HP`, `AC`, `STR`, `DEX`, `CON`, `INT`, `WIS`, `CHA`, and `CR`. 
* **Action:** We prioritize features with high coefficients ($> 0.7$) as they likely represent the strongest predictors for the model.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define features
cols = ['HP', 'AC', 'Strength', 'Dexterity', 'Constitution', 
        'Intelligence', 'Wisdom', 'Charisma', 'CR']

# Calculate correlation
corr = prepared_df[cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation of Stats with Challenge Rating')
plt.show()

# Print results for each feature relative to CR
print("--- Correlation Results (Target: CR) ---")
for feature in cols[:-1]: # Exclude CR itself
    coef = corr.loc[feature, 'CR']
    status = "STRONG" if abs(coef) > 0.7 else "WEAK"
    print(f"{feature}: {coef:.2f} ({status})")

This heatmap shows exactly which stats drive my monsters' power levels. Hit Points (0.93) and Constitution (0.78) are the only two stats with a strong correlation to Challenge Rating. This confirms that in my dataset, a monster's survivability is the most important factor in determining its rank.

My suspicion about scaling is likely focused on the weak stats like Dexterity (0.18) or Intelligence (0.50). While these numbers are lower, it does not mean they are scaled incorrectly in my data. It means they are not reliable predictors of power. A high-CR monster in my list can be just as clumsy as a low-CR monster, but it almost always has more health.

The high correlation for HP (0.93) is actually a warning for my future model. Because HP and CR move almost perfectly together, a simple linear model might rely too heavily on HP and ignore everything else. This could cause issues if I try to predict the CR of a "glass cannon" monster that has low health but very high damage.

### 2.2 Scatter Plots (Linearity Check)
* **HP vs. CR:** We examine if `HP` increases linearly with `CR` or if it scales exponentially at higher tiers (CR 20+).
* **Enhanced Scatter:** We plot `HP` (x-axis) vs. `CR` (y-axis) using `Size` as a color hue. This helps determine if a "Large" monster at a specific CR tier has significantly different health pools than a "Small" monster at the same tier.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot HP vs CR
plt.figure(figsize=(10, 6))
sns.regplot(data=prepared_df, x='HP', y='CR', 
            scatter_kws={'alpha':0.5, 'color':'teal'}, 
            line_kws={'color':'red'})

plt.title('Linearity Check: HP vs Challenge Rating')
plt.xlabel('Hit Points')
plt.ylabel('Challenge Rating')
plt.show()

# Print growth check
low_cr_avg = prepared_df[prepared_df['CR'] <= 5]['HP'].mean()
high_cr_avg = prepared_df[prepared_df['CR'] >= 20]['HP'].mean()

print(f"Average HP at CR 0-5: {low_cr_avg:.1f}")
print(f"Average HP at CR 20+: {high_cr_avg:.1f}")
print(f"Growth Factor: {high_cr_avg / low_cr_avg:.1f}x")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define size order for the legend
size_order = ['tiny', 'small', 'medium', 'large', 'huge', 'gargantuan']

# Plot HP vs CR with Size as hue
plt.figure(figsize=(12, 7))
sns.scatterplot(data=prepared_df, x='HP', y='CR', hue='Size', 
                hue_order=size_order, palette='viridis', alpha=0.7)

plt.title('HP vs Challenge Rating by Monster Size')
plt.xlabel('Hit Points')
plt.ylabel('Challenge Rating')
plt.legend(title='Monster Size', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Print average HP per size category to compare
print("--- Average HP by Size ---")
print(prepared_df.groupby('Size')['HP'].mean().reindex(size_order))

The scatter plots reveal a direct link between monster health and rank. The red trend line stays tight to the data points which proves that Hit Points and Challenge Rating move together in a nearly straight line. HP serves as the primary tool for setting monster difficulty levels.

The growth check numbers show a massive jump in power at higher tiers. Monsters at CR 20 or higher have nearly 10 times the average health of those at CR 5. This growth factor indicates that late game threats rely on massive health pools to challenge players.

The size coded plot shows that physical scale is a major factor. Tiny and Small monsters are almost all clustered in the bottom corner with low HP and low CR. Huge and Gargantuan monsters are the only ones capable of reaching the top tier.

The average HP by size table supports this trend. A Gargantuan monster has nearly 30 times more health than a Tiny one on average. Size acts as a gatekeeper in this data. To reach a high Challenge Rating a monster almost always has to be Huge or Gargantuan.

## Phase 3: Categorical Analysis (The "Why")
**Goal:** Determine if a monster's category (Size or Type) inherently forces it into a higher CR bracket.

### 3.1 Size vs. CR
* **Boxplot Analysis:** We group `CR` by `Size` to test the hypothesis that larger creatures generally occupy higher power tiers.
* **Observation:** We look for "weak" Gargantuan creatures or "powerful" Tiny creatures that may act as outliers to the general rule.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define logical size order
size_order = ['tiny', 'small', 'medium', 'large', 'huge', 'gargantuan']

# Create boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(data=prepared_df, x='Size', y='CR', order=size_order, palette='magma')
plt.title('CR Distribution by Monster Size')
plt.show()

# Print statistical summary
print("--- CR Statistics by Size ---")
stats = prepared_df.groupby('Size')['CR'].agg(['mean', 'median', 'max']).reindex(size_order)
print(stats)

# Identify outliers
print("\n--- Outlier Detection ---")
powerful_tiny = prepared_df[(prepared_df['Size'] == 'tiny') & (prepared_df['CR'] > 5)]
print(f"High-CR Tiny Monsters: {len(powerful_tiny)}")
if not powerful_tiny.empty:
    print(powerful_tiny[['Name', 'CR']].sort_values(by='CR', ascending=False).head())

weak_gargantuan = prepared_df[(prepared_df['Size'] == 'gargantuan') & (prepared_df['CR'] < 10)]
print(f"\nLow-CR Gargantuan Monsters: {len(weak_gargantuan)}")
if not weak_gargantuan.empty:
    print(weak_gargantuan[['Name', 'CR']].sort_values(by='CR').head())

In [ ]:
# Identify outliers at the extremes of size
print("--- Powerful Tiny Monsters (CR > 5) ---")
high_power_tiny = prepared_df[(prepared_df['Size'] == 'tiny') & (prepared_df['CR'] > 5)]
print(high_power_tiny[['Name', 'CR', 'Type']].sort_values(by='CR', ascending=False))

print("\n--- Weak Gargantuan Monsters (CR < 10) ---")
low_power_gargantuan = prepared_df[(prepared_df['Size'] == 'gargantuan') & (prepared_df['CR'] < 10)]
print(low_power_gargantuan[['Name', 'CR', 'Type']].sort_values(by='CR'))

# Visualization of these outliers
plt.figure(figsize=(10, 6))
sns.stripplot(data=prepared_df, x='Size', y='CR', order=size_order, jitter=0.3, alpha=0.3, color='gray')
sns.scatterplot(data=high_power_tiny, x='Size', y='CR', color='red', label='Powerful Tiny')
sns.scatterplot(data=low_power_gargantuan, x='Size', y='CR', color='blue', label='Weak Gargantuan')
plt.title('Identifying Size-Power Outliers')
plt.legend()
plt.show()

The boxplot analysis confirms that monster size and power level are closely linked. As size increases from Tiny to Gargantuan, the average Challenge Rating rises steadily. Tiny creatures stay near the bottom of the scale with a median CR close to zero. Gargantuan creatures represent a massive jump in power with a median CR of 21.

The outlier detection identifies rare exceptions to this rule. A few Tiny monsters like the Demilich reach very high CR levels despite their size. These outliers suggest that high intelligence or magical abilities can override the usual physical limits of a small body.

Low-power Gargantuan monsters also exist but are less common. Creatures like the Brontosaurus have high health but low CR because they lack dangerous attacks or high armor. This confirms that while size usually dictates a monster's potential, specific abilities or lack of defenses can push a creature far away from its expected power tier.

### 3.2 Type vs. CR
* **Mean CR per Type:** We calculate the average `CR` for each monster `Type` using a bar chart.
* **Comparison:** We compare high-fantasy archetypes (e.g., "Dragons") against baseline creatures (e.g., "Beasts") to see if `Type` is a statistically significant feature.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate mean CR for each type and sort
type_cr = prepared_df.groupby('Type')['CR'].mean().sort_values(ascending=False)

# Plot bar chart
plt.figure(figsize=(10, 8))
sns.barplot(x=type_cr.values, y=type_cr.index, palette='coolwarm')
plt.title('Average Challenge Rating by Monster Type')
plt.xlabel('Mean Challenge Rating')
plt.ylabel('Creature Type')
plt.show()

# Print results
print("Mean CR by Type:")
print(type_cr)

In [ ]:
# Get all unique types and calculate mean CR
all_types = prepared_df.groupby('Type')['CR'].agg(['mean', 'count']).sort_values('mean', ascending=False)

# Plotting the full distribution
plt.figure(figsize=(12, 8))
sns.boxplot(data=prepared_df, x='CR', y='Type', order=all_types.index, palette='vlag')
plt.title('CR Distribution Across All Creature Types')
plt.xlabel('Challenge Rating')
plt.show()

# Print the full list for comparison
print("--- Full Type Comparison (Sorted by Power) ---")
print(all_types)

# Comparison logic: Top vs Bottom
top_type = all_types.index[0]
bottom_type = all_types.index[-1]
gap = all_types.loc[top_type, 'mean'] / all_types.loc[bottom_type, 'mean']

print(f"\nMost powerful type: {top_type} (Mean CR: {all_types.loc[top_type, 'mean']:.2f})")
print(f"Least powerful type: {bottom_type} (Mean CR: {all_types.loc[bottom_type, 'mean']:.2f})")
print(f"Power gap: {gap:.1f}x difference in average CR.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate counts for each type and sort from high to low
type_ranks = prepared_df['Type'].value_counts()

# Plot the distribution
plt.figure(figsize=(12, 8))
sns.barplot(x=type_ranks.values, y=type_ranks.index, palette='mako')
plt.title('Monster Types Ranked by Frequency')
plt.xlabel('Number of Monsters')
plt.ylabel('Creature Type')
plt.show()

# Print the ranked results
print("--- Monster Types Ranked by Count ---")
print(type_ranks)

The full distribution results show a massive power hierarchy based on creature type. Celestial-fiend hybrids and dragons sit at the top of the range. Swarms and tiny constructs occupy the very bottom. These rankings demonstrate that specific labels are strong indicators of a monster's role in the game.

The gap between the highest and lowest tiers is extreme. The average Challenge Rating for the most powerful group is over 49 times higher than the average for a swarm of tiny constructs. This power gap highlights that type is a major predictor for the ranking system.

Many types like fiends and celestials show a wide spread of power levels in the boxplot. This means they appear at various stages of gameplay. Other types like beasts or swarms are tightly packed at the lower end of the scale. These results confirm that including the full variety of types provides critical information for distinguishing between weak encounters and epic bosses.

## Summary 

---

The bestiary dataset is heavily skewed toward low power levels. Most monsters cluster between Challenge Rating 0 and 5 while legendary threats are rare. Defensive stats show two different patterns. Armor Class is restricted to a narrow range making every single point highly valuable. Hit Points are unbounded and scale dramatically reaching over 700 for top tier bosses. Physical attributes like Strength and Constitution show more variety and higher averages than mental stats. This makes them better predictors of monster power.

Categorical data shows that Medium is the standard size for most creatures. Humanoids and monstrosities make up the bulk of the population. There is a clear hierarchy where size and type dictate potential. Larger monsters almost always have higher ranks. Dragons and celestials sit at the top of the power scale while beasts and swarms remain at the bottom. Rare outliers like high power tiny creatures suggest that magical traits can sometimes override physical size.

Correlation analysis confirms that survivability is the main driver of Challenge Rating. Hit Points and Constitution move in close sync with rank. Other stats like Dexterity and Wisdom remain consistent across all power levels and offer less predictive value. The strong linear relationship between health and rank proves that Hit Points are the primary tool used to scale game difficulty. This suggests the model should focus on health and physical size to accurately estimate monster strength.